## Olist E-Commerce: Data Exploration with DuckDB

## 1. Setup the notebook

In this notebook we load five tables of the e-commerce dataset into DuckDB and explore them with SQL before designing a data warehouse.

**What we will do**

1. Load the CSV files into DuckDB
2. Understand each table: structure, grain, keys
3. Check the relationships (foreign keys)
4. Find data quality issues
5. Build a star schema

**Tables:** `customers`, `orders`, `order_items`, `products`, `sellers`

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
WAREHOUSE_DIR = DATA_DIR / "warehouse"
csv = RAW_DIR / "olist_orders_dataset.csv"

In [3]:
print(duckdb.sql(f"SELECT count(*) FROM '{csv}'"))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘



### DuckDB lets query CSV data with SQL. The database file persists locally so later notebooks can reuse the loaded raw tables.

In [4]:
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

In [5]:
def q(sql_str: str) -> pd.DataFrame:
    """Run a SQL query in DuckDB and return a pandas DataFrame."""
    return con.sql(sql_str).df()

## 2. Load the source tables into a **raw schema**

A Schema is a namespace that groups tables. We keep the source copies in **raw**
so that they remain distinguishable from cleaned warehouse tables we will create later.

### Define a mapping to transform column types

In [6]:
RAW_TABLES: dict[str, dict[str, str]] = {
    "customers": {"customer_zip_code_prefix": "VARCHAR"},  # keep leading zeros
    "orders": {},
    "order_items": {},
    "products": {},
    "sellers": {}
}

### Define some helper functions

In [7]:
def csv_path(table: str) -> Path:
    """"""
    return RAW_DIR / f"olist_{table}_dataset.csv"

def check_source_files(tables) -> None:
    """Fail once, list every missing file"""
    missing = [csv_path(t) for t in tables if not csv_path(t).exists()]
    if missing:
        names = "\n".join(f" - {p}" for p in missing)
        raise FileNotFoundError(f"Missing CSV files: {names}")

#### Define schema name for source data

In [8]:
RAW_SCHEMA = "raw"

### Load data into source tables

In [9]:
def load_raw_table(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"DROP TABLE IF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

### Sanity check if we have all required source files

In [10]:
check_source_files(RAW_TABLES)

### Create **RAW_SCHEMA** if it does not exist

In [11]:
con.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")

### Execute **load_raw_table** to load all source data

In [12]:
row_counts = {t: load_raw_table(con, t, types) for t, types in RAW_TABLES.items()}

### Check count of rows inserted into each `raw`.`<table>`

In [13]:
pd.DataFrame(row_counts.items(), columns=["table_name", "row_count"])

,table_name,row_count
0,customers,99441
1,orders,99441
2,order_items,112650
3,products,32951
4,sellers,3095


## 3. Look at each table

For each raw table, inspect its columns with `DESCRIBE` and its first five rows. The grain (one row per) is:

- `customers`: one customer-address record (`customer_id`); a shopper can have multiple records under `customer_unique_id`.
- `orders`: one order.
- `order_items`: one product line within an order; `order_item_id` is the line number within that order.
- `products`: one catalog product.
- `sellers`: one seller.

In [14]:
for t in RAW_TABLES:
    print(f"----- {t} -----")
    display(q(f"DESCRIBE raw.{t}")[["column_name", "column_type", "null"]])
    display(q(f"SELECT * FROM raw.{t} LIMIT 5"))

----- customers -----


,column_name,column_type,null
0,customer_id,VARCHAR,YES
1,customer_unique_id,VARCHAR,YES
2,customer_zip_code_prefix,VARCHAR,YES
3,customer_city,VARCHAR,YES
4,customer_state,VARCHAR,YES


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


----- orders -----


,column_name,column_type,null
0,order_id,VARCHAR,YES
1,customer_id,VARCHAR,YES
2,order_status,VARCHAR,YES
3,order_purchase_timestamp,TIMESTAMP,YES
4,order_approved_at,TIMESTAMP,YES
5,order_delivered_carrier_date,TIMESTAMP,YES
6,order_delivered_customer_date,TIMESTAMP,YES
7,order_estimated_delivery_date,TIMESTAMP,YES


...

,column_name,column_type,null
0,product_id,VARCHAR,YES
1,product_category_name,VARCHAR,YES
2,product_name_lenght,BIGINT,YES
3,product_description_lenght,BIGINT,YES
4,product_photos_qty,BIGINT,YES
5,product_weight_g,BIGINT,YES
6,product_length_cm,BIGINT,YES
7,product_height_cm,BIGINT,YES
8,product_width_cm,BIGINT,YES


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


----- sellers -----


,column_name,column_type,null
0,seller_id,VARCHAR,YES
1,seller_zip_code_prefix,VARCHAR,YES
2,seller_city,VARCHAR,YES
3,seller_state,VARCHAR,YES


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


`SUMMARIZE`profiles every column is one command: min, max, average, null percentage,
approximate distinct count.

In [15]:
q("SUMMARIZE raw.order_items")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,87618,NaN,NaN,NaN,NaN,NaN,112650,0.0
1,order_item_id,BIGINT,1,21,21,1.1978339991122948,0.7051240313951734,1,1,1,112650,0.0
2,product_id,VARCHAR,00066f42aeeb9f3007548bb9d3f33c38,fffe9eeff12fcbd74a2f2b007dde0c58,24292,NaN,NaN,NaN,NaN,NaN,112650,0.0
3,seller_id,VARCHAR,0015a82c2db000af6aaaf3ae2ecb0532,ffff564a4f9085cd26170f4732393726,3886,NaN,NaN,NaN,NaN,NaN,112650,0.0
4,shipping_limit_date,TIMESTAMP,2016-09-19 00:15:34,2020-04-09 22:35:08,80774,2018-01-07 15:36:52.192685,NaN,2017-09-20 14:24:06.919554,2018-01-27 19:47:01.503972,2018-05-11 00:22:53.928243,112650,0.0
5,price,DOUBLE,0.85,6735.0,6497,120.65373901477311,183.6339280502597,40.01666999532123,74.5760959518914,134.19097071958893,112650,0.0
6,freight_value,DOUBLE,0.0,409.68,5665,19.99031992898562,15.806405412296998,13.071531156591233,16.279508106244663,21.16306476327699,112650,0.0


In [16]:
q("SUMMARIZE raw.order_items")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,87618,NaN,NaN,NaN,NaN,NaN,112650,0.0
1,order_item_id,BIGINT,1,21,21,1.1978339991122948,0.7051240313951734,1,1,1,112650,0.0
2,product_id,VARCHAR,00066f42aeeb9f3007548bb9d3f33c38,fffe9eeff12fcbd74a2f2b007dde0c58,24292,NaN,NaN,NaN,NaN,NaN,112650,0.0
3,seller_id,VARCHAR,0015a82c2db000af6aaaf3ae2ecb0532,ffff564a4f9085cd26170f4732393726,3886,NaN,NaN,NaN,NaN,NaN,112650,0.0
4,shipping_limit_date,TIMESTAMP,2016-09-19 00:15:34,2020-04-09 22:35:08,80774,2018-01-07 15:36:52.192685,NaN,2017-09-20 14:24:06.919554,2018-01-27 19:47:01.503972,2018-05-11 00:22:53.928243,112650,0.0
5,price,DOUBLE,0.85,6735.0,6497,120.65373901477311,183.6339280502597,40.01666999532123,74.5760959518914,134.19097071958893,112650,0.0
6,freight_value,DOUBLE,0.0,409.68,5665,19.99031992898562,15.806405412296998,13.071531156591233,16.279508106244663,21.16306476327699,112650,0.0


In [17]:
q("SUMMARIZE raw.products")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,product_id,VARCHAR,00066f42aeeb9f3007548bb9d3f33c38,fffe9eeff12fcbd74a2f2b007dde0c58,24292,NaN,NaN,NaN,NaN,NaN,32951,0.00
1,product_category_name,VARCHAR,agro_industria_e_comercio,utilidades_domesticas,66,NaN,NaN,NaN,NaN,NaN,32951,1.85
2,product_name_lenght,BIGINT,5,76,64,48.47694876472589,10.245740725237287,42,51,57,32951,1.85
3,product_description_lenght,BIGINT,4,3992,2781,771.4952846232337,635.1152246349538,339,594,973,32951,1.85
4,product_photos_qty,BIGINT,1,20,20,2.1889861166939797,1.7367656379315435,1,1,3,32951,1.85
5,product_weight_g,BIGINT,0,40425,2155,2276.4724877841513,4282.038730977024,300,701,1883,32951,0.01
6,product_length_cm,BIGINT,7,105,97,30.81507784758263,16.914458054065953,18,25,39,32951,0.01
7,product_height_cm,BIGINT,2,105,99,16.937661234028347,13.637554061749569,8,13,21,32951,0.01
8,product_width_cm,BIGINT,6,118,92,23.196728277034204,12.079047453227794,15,20,30,32951,0.01


## 4. Primary Keys

A **primary key** uniquely identifies a row. A column is a valid candidate only if `count(*) = count(distinct ...)` and it has no nulls.

- A **composite key** uses several columns together (`order_items`)
- A **natural key** comes from the business (`customer_unique_id`).
- A **surrogate key** is generated by the warehouse (`customer_key = 1, 2, 3...`)

In [18]:
q("""
WITH candidate_keys AS (
    SELECT 'customers.customer_id' AS candidate, count(*) AS row_count,
           count(DISTINCT customer_id) AS distinct_count,
           count(*) FILTER (WHERE customer_id IS NULL) AS null_count
    FROM raw.customers
    UNION ALL
    SELECT 'customers.customer_unique_id', count(*), count(DISTINCT customer_unique_id),
           count(*) FILTER (WHERE customer_unique_id IS NULL)
    FROM raw.customers
    UNION ALL
    SELECT 'orders.order_id', count(*), count(DISTINCT order_id),
           count(*) FILTER (WHERE order_id IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'order_items.order_id (alone)', count(*), count(DISTINCT order_id),
           count(*) FILTER (WHERE order_id IS NULL)
    FROM raw.order_items
    UNION ALL
    SELECT 'order_items (order_id, order_item_id)', count(*),
           count(DISTINCT (order_id, order_item_id)),
           count(*) FILTER (WHERE order_id IS NULL OR order_item_id IS NULL)
    FROM raw.order_items
    UNION ALL
    SELECT 'products.product_id', count(*), count(DISTINCT product_id),
           count(*) FILTER (WHERE product_id IS NULL)
    FROM raw.products
    UNION ALL
    SELECT 'sellers.seller_id', count(*), count(DISTINCT seller_id),
           count(*) FILTER (WHERE seller_id IS NULL)
    FROM raw.sellers
)
SELECT *, row_count = distinct_count AND null_count = 0 AS is_valid_key
FROM candidate_keys
""")

,candidate,row_count,distinct_count,null_count,is_valid_key
0,customers.customer_id,99441,99441,0,True
1,customers.customer_unique_id,99441,96096,0,False
2,orders.order_id,99441,99441,0,True
3,order_items.order_id (alone),112650,98666,0,False
4,"order_items (order_id, order_item_id)",112650,112650,0,True
5,products.product_id,32951,32951,0,True
6,sellers.seller_id,3095,3095,0,True


## 4. Foreign keys and relationships

Each relationship below is checked from the child to the parent using an `ANTI JOIN`; its output also measures child rows per parent. A composite primary key in `order_items` makes that table the line-item junction between orders, products, and sellers.

In [ ]:
q("""
WITH relationship_checks AS (
    SELECT 'customers.customer_id -> orders.customer_id' AS relationship,
           (SELECT count(*) FROM raw.orders o ANTI JOIN raw.customers c
            ON o.customer_id = c.customer_id) AS orphan_child_rows,
           (SELECT count(*) FROM raw.orders WHERE customer_id IS NULL) AS null_fk_rows,
           (SELECT min(child_count) FROM (
                SELECT c.customer_id, count(o.customer_id) AS child_count
                FROM raw.customers c LEFT JOIN raw.orders o
                    ON o.customer_id = c.customer_id GROUP BY c.customer_id
            )) AS min_children_per_parent,
           (SELECT max(child_count) FROM (
                SELECT c.customer_id, count(o.customer_id) AS child_count
                FROM raw.customers c LEFT JOIN raw.orders o
                    ON o.customer_id = c.customer_id GROUP BY c.customer_id
            )) AS max_children_per_parent,
           (SELECT count(*) FROM (
                SELECT c.customer_id, count(o.customer_id) AS child_count
                FROM raw.customers c LEFT JOIN raw.orders o
                    ON o.customer_id = c.customer_id GROUP BY c.customer_id
                HAVING count(o.customer_id) = 0
            )) AS parents_without_children
    UNION ALL
    SELECT 'orders.order_id -> order_items.order_id',
           (SELECT count(*) FROM raw.order_items i ANTI JOIN raw.orders o
            ON i.order_id = o.order_id),
           (SELECT count(*) FROM raw.order_items WHERE order_id IS NULL),
           (SELECT min(child_count) FROM (
                SELECT o.order_id, count(i.order_id) AS child_count
                FROM raw.orders o LEFT JOIN raw.order_items i
                    ON i.order_id = o.order_id GROUP BY o.order_id
            )),
           (SELECT max(child_count) FROM (
                SELECT o.order_id, count(i.order_id) AS child_count
                FROM raw.orders o LEFT JOIN raw.order_items i
                    ON i.order_id = o.order_id GROUP BY o.order_id
            )),
           (SELECT count(*) FROM (
                SELECT o.order_id, count(i.order_id) AS child_count
                FROM raw.orders o LEFT JOIN raw.order_items i
                    ON i.order_id = o.order_id GROUP BY o.order_id
                HAVING count(i.order_id) = 0
            ))
    UNION ALL
    SELECT 'products.product_id -> order_items.product_id',
           (SELECT count(*) FROM raw.order_items i ANTI JOIN raw.products p
            ON i.product_id = p.product_id),
           (SELECT count(*) FROM raw.order_items WHERE product_id IS NULL),
           (SELECT min(child_count) FROM (
                SELECT p.product_id, count(i.product_id) AS child_count
                FROM raw.products p LEFT JOIN raw.order_items i
                    ON i.product_id = p.product_id GROUP BY p.product_id
            )),
           (SELECT max(child_count) FROM (
                SELECT p.product_id, count(i.product_id) AS child_count
                FROM raw.products p LEFT JOIN raw.order_items i
                    ON i.product_id = p.product_id GROUP BY p.product_id
            )),
           (SELECT count(*) FROM (
                SELECT p.product_id, count(i.product_id) AS child_count
                FROM raw.products p LEFT JOIN raw.order_items i
                    ON i.product_id = p.product_id GROUP BY p.product_id
                HAVING count(i.product_id) = 0
            ))
    UNION ALL
    SELECT 'sellers.seller_id -> order_items.seller_id',
           (SELECT count(*) FROM raw.order_items i ANTI JOIN raw.sellers s
            ON i.seller_id = s.seller_id),
           (SELECT count(*) FROM raw.order_items WHERE seller_id IS NULL),
           (SELECT min(child_count) FROM (
                SELECT s.seller_id, count(i.seller_id) AS child_count
                FROM raw.sellers s LEFT JOIN raw.order_items i
                    ON i.seller_id = s.seller_id GROUP BY s.seller_id
            )),
           (SELECT max(child_count) FROM (
                SELECT s.seller_id, count(i.seller_id) AS child_count
                FROM raw.sellers s LEFT JOIN raw.order_items i
                    ON i.seller_id = s.seller_id GROUP BY s.seller_id
            )),
           (SELECT count(*) FROM (
                SELECT s.seller_id, count(i.seller_id) AS child_count
                FROM raw.sellers s LEFT JOIN raw.order_items i
                    ON i.seller_id = s.seller_id GROUP BY s.seller_id
                HAVING count(i.seller_id) = 0
            ))
)
SELECT * FROM relationship_checks
""")

No orphans in the child-to-parent direction. But 775 orders have no items. An **INNER JOIN** from orders
to items would silently remove them.

## 5. Data Quality

we check five dimensions: completeness(nulls), uniqueness, validity (values in a sensible range),
consistency (fields agree), timeliness (coverage)

### 5.1 Orders with no items

#### Exercise 1

In [19]:
q("""
SELECT o.order_status, count(*) AS orders_without_items
FROM raw.orders o
ANTI JOIN raw.order_items i USING (order_id)
GROUP BY o.order_status
ORDER BY orders_without_items DESC, o.order_status
""")

,order_status,orders_without_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


### 5.2 Check order status and class imbalance

#### Exercise 2

In [20]:
q("""
SELECT order_status, count(*) AS order_count,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct_of_orders
FROM raw.orders
GROUP BY order_status
ORDER BY order_count DESC, order_status
""")

,order_status,order_count,pct_of_orders
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


### 5.3 Missing and inconsistent timestamps

#### Exercise 3

In [21]:
q("""
SELECT check_name, rows_flagged
FROM (
    SELECT 'purchase timestamp is null' AS check_name,
           count(*) FILTER (WHERE order_purchase_timestamp IS NULL) AS rows_flagged
    FROM raw.orders
    UNION ALL
    SELECT 'approval timestamp is null', count(*) FILTER (WHERE order_approved_at IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'carrier timestamp is null', count(*) FILTER (WHERE order_delivered_carrier_date IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'customer delivery timestamp is null', count(*) FILTER (WHERE order_delivered_customer_date IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'estimated delivery timestamp is null', count(*) FILTER (WHERE order_estimated_delivery_date IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'delivered orders have no customer delivery timestamp',
           count(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'carrier date is before purchase date',
           count(*) FILTER (WHERE order_delivered_carrier_date < order_purchase_timestamp)
    FROM raw.orders
    UNION ALL
    SELECT 'customer delivery date is before carrier date',
           count(*) FILTER (WHERE order_delivered_customer_date < order_delivered_carrier_date)
    FROM raw.orders
    UNION ALL
    SELECT 'actual delivery is later than estimate',
           count(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date)
    FROM raw.orders
) AS timestamp_checks
ORDER BY check_name
""")

,check_name,rows_flagged
0,actual delivery is later than estimate,7827
1,approval timestamp is null,160
2,carrier date is before purchase date,166
3,carrier timestamp is null,1783
4,customer delivery date is before carrier date,23
5,customer delivery timestamp is null,2965
6,delivered orders have no customer delivery tim...,8
7,estimated delivery timestamp is null,0
8,purchase timestamp is null,0


### 5.4 Time coverage

#### Exercise 4

In [25]:
monthly = q("""
SELECT strftime(date_trunc('month', order_purchase_timestamp), '%Y-%m') AS "month",
       count(*) AS orders
FROM raw.orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY 1
ORDER BY 1
""")

### 5.5 Product completeness check

#### Exercise 5

In [23]:
q("""
SELECT column_name, row_count, null_count,
       round(100.0 * null_count / row_count, 2) AS pct_missing
FROM (
    SELECT 'product_id' AS column_name, count(*) AS row_count,
           count(*) FILTER (WHERE product_id IS NULL) AS null_count FROM raw.products
    UNION ALL
    SELECT 'product_category_name', count(*), count(*) FILTER (WHERE product_category_name IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_name_lenght', count(*), count(*) FILTER (WHERE product_name_lenght IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_description_lenght', count(*), count(*) FILTER (WHERE product_description_lenght IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_photos_qty', count(*), count(*) FILTER (WHERE product_photos_qty IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_weight_g', count(*), count(*) FILTER (WHERE product_weight_g IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_length_cm', count(*), count(*) FILTER (WHERE product_length_cm IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_height_cm', count(*), count(*) FILTER (WHERE product_height_cm IS NULL) FROM raw.products
    UNION ALL
    SELECT 'product_width_cm', count(*), count(*) FILTER (WHERE product_width_cm IS NULL) FROM raw.products
) AS product_completeness
ORDER BY null_count DESC, column_name
""")

,column_name,row_count,null_count,pct_missing
0,product_category_name,32951,610,1.85
1,product_description_lenght,32951,610,1.85
2,product_name_lenght,32951,610,1.85
3,product_photos_qty,32951,610,1.85
4,product_height_cm,32951,2,0.01
5,product_length_cm,32951,2,0.01
6,product_weight_g,32951,2,0.01
7,product_width_cm,32951,2,0.01
8,product_id,32951,0,0.00
